# BERT 职能分类器（Kaggle GPU 版）

归口预测第二级：留言 → 职能类别。5 折交叉验证。

**使用前**：把 `data/labeled_1200.csv` 上传为 Kaggle Dataset（例如名为 `liuyanban1200`），然后修改下方 `DATA` 路径。

In [ ]:
!pip install -q transformers scikit-learn 2>/dev/null || true
import os, time
import numpy as np, pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import BertForSequenceClassification, BertTokenizerFast
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

In [ ]:
_RULES = [
    # ---- 教育体育 ----
    ("教育体育局", "教育体育"), ("教体局", "教育体育"), ("教育局", "教育体育"), ("教委", "教育体育"),
    ("教育厅", "教育体育"), ("体育局", "教育体育"), ("体育", "教育体育"),
    # ---- 住建 ----
    ("住房和城乡", "住建"), ("住建", "住建"), ("城乡建设", "住建"), ("房管", "住建"), ("住房", "住建"),
    ("建设局", "住建"), ("建委", "住建"), ("规划", "住建"), ("自然资源", "住建"), ("国土", "住建"), ("不动产", "住建"),
    # ---- 城管执法 ----
    ("城管", "城管执法"), ("市容", "城管执法"), ("环卫", "城管执法"), ("行政执法", "城管执法"), ("城乡管理", "城管执法"),
    # ---- 生态环境 ----
    ("生态环境", "生态环境"), ("环保", "生态环境"), ("生态", "生态环境"),
    # ---- 公安司法 ----
    ("公安", "公安司法"), ("交警", "公安司法"), ("派出所", "公安司法"), ("司法", "公安司法"),
    ("法院", "公安司法"), ("检察", "公安司法"), ("消防", "公安司法"),
    # ---- 交通 ----
    ("交通", "交通"), ("运输", "交通"), ("公路", "交通"),
    # ---- 市场监管 ----
    ("市场监督", "市场监管"), ("市场监管", "市场监管"), ("工商", "市场监管"), ("食药", "市场监管"), ("质监", "市场监管"),
    # ---- 卫生健康 ----
    ("卫健委", "卫生健康"), ("卫生", "卫生健康"), ("医保", "卫生健康"), ("医院", "卫生健康"), ("疾控", "卫生健康"),
    # ---- 人社 ----
    ("人社", "人社"), ("劳动", "人社"), ("就业", "人社"), ("社保", "人社"),
    # ---- 民政 ----
    ("民政", "民政"), ("残联", "民政"), ("退役", "民政"),
    # ---- 水务 ----
    ("水务", "水务"), ("水利", "水务"), ("供水", "水务"), ("自来水", "水务"),
    # ---- 农业农村 ----
    ("农业", "农业农村"), ("农村", "农业农村"), ("畜牧", "农业农村"), ("乡村", "农业农村"),
    # ---- 发改经信 ----
    ("发改", "发改经信"), ("经信", "发改经信"), ("工信", "发改经信"), ("商务", "发改经信"), ("物价", "发改经信"),
    # ---- 应急管理 ----
    ("应急", "应急管理"), ("安监", "应急管理"), ("安全生产", "应急管理"),
    # ---- 文旅 ----
    ("文旅", "文旅"), ("文化", "文旅"), ("旅游", "文旅"), ("文物", "文旅"),
    # ---- 政务热线/办理机构 ----
    ("12345", "政务热线"), ("留言办理", "政务热线"), ("热线", "政务热线"),
    ("市长公开电话", "政务热线"), ("政务服务中心", "政务热线"), ("政务中心", "政务热线"),
    ("回复组", "政务热线"), ("营商", "政务热线"), ("便民", "政务热线"), ("政务热线", "政务热线"),
    # ---- 能源 ----
    ("电力", "能源"), ("供电", "能源"), ("电网", "能源"), ("燃气", "能源"), ("热力", "能源"), ("供暖", "能源"), ("能源", "能源"),
    # ---- 街道乡镇 ----
    ("街道办事处", "街道乡镇"), ("街道", "街道乡镇"), ("乡镇", "街道乡镇"), ("社区", "街道乡镇"),
    ("镇政府", "街道乡镇"), ("镇", "街道乡镇"),
    # ---- 财政税务 ----
    ("财政", "财政税务"), ("税务", "财政税务"), ("金融", "财政税务"),
    # ---- 通信邮政 ----
    ("邮政", "通信邮政"), ("通信", "通信邮政"), ("电信", "通信邮政"), ("移动", "通信邮政"),
    # ---- 物业房产 ----
    ("物业", "物业房产"), ("公积金", "物业房产"),
    # ---- 党委政府办 / 行政区划（兜底，顺序靠后）----
    ("委办", "党委政府办"), ("党办", "党委政府办"), ("政府办", "党委政府办"), ("人民政府", "党委政府办"),
    ("办公厅", "党委政府办"), ("督查", "党委政府办"), ("党委", "党委政府办"),
    ("市委", "党委政府办"), ("区委", "党委政府办"), ("县委", "党委政府办"),
    ("信访", "党委政府办"), ("书记", "党委政府办"), ("市长", "党委政府办"), ("区长", "党委政府办"),
    ("新区", "党委政府办"), ("开发区", "党委政府办"), ("高新区", "党委政府办"), ("园区", "党委政府办"), ("管委会", "党委政府办"),
    ("政府", "党委政府办"),
    ("区", "党委政府办"), ("县", "党委政府办"), ("市", "党委政府办"),
]

def map_department(name):
    for kw, cls in _RULES:
        if kw in name:
            return cls
    return "其他"


In [ ]:

# ==== 自动探测挂载的数据集（无需手改路径）====
# 若报错，请在右侧 "Add Input" 里搜索你上传的 dataset 并挂载，然后重跑本 cell
import glob
_csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("挂载到的 CSV 文件:", _csvs)
assert _csvs, "未找到 CSV！请点击右侧 Add Input 挂载 labeled_1200.csv 所在的数据集"
DATA = next((p for p in _csvs if "labeled_1200" in p), _csvs[0])
print("使用数据:", DATA)

MODEL_NAME = "bert-base-chinese"   # Kaggle 直接从 HF 下载；网络慢可换 hf-mirror
EPOCHS = 8
FOLDS = 5
LR = 2e-5
MAX_LEN = 128
MIN_CLASS = 10

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()} | {"labels": self.labels[i]}

@torch.no_grad()
def evaluate(model, ds, device):
    model.eval()
    loader = DataLoader(ds, batch_size=64)
    pred, label = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        pred.extend(logits.argmax(-1).cpu().numpy().tolist())
        label.extend(batch["labels"].cpu().numpy().tolist())
    return accuracy_score(label, pred), f1_score(label, pred, average="macro", zero_division=0)

def train_fold(model, tokenizer, tr_ds, va_ds, device, class_weight):
    loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    best_f1, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = torch.nn.functional.cross_entropy(out.logits, batch["labels"], weight=class_weight)
            opt.zero_grad(); loss.backward(); opt.step()
        acc, f1 = evaluate(model, va_ds, device)
        print(f"  ep{ep+1} val_acc={acc:.4f} macroF1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return best_f1

# 数据准备
df = pd.read_csv(DATA, encoding="utf-8-sig", dtype=str).fillna("")
df["回复组织"] = df["回复组织"].str.strip()
df = df[df["回复组织"] != ""].copy()
df["职能"] = df["回复组织"].map(map_department)
vc = df["职能"].value_counts()
small = set(vc[vc < MIN_CLASS].index)
df.loc[df["职能"].isin(small), "职能"] = "其他"
classes = sorted(df["职能"].unique())
cls2id = {c: i for i, c in enumerate(classes)}
print(f"samples={len(df)} classes={len(classes)}")
print(classes)

texts = (df["留言内容"] + "\n" + df["demand"]).tolist()
labels = [cls2id[c] for c in df["职能"]]
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for fold, (tr, va) in enumerate(skf.split(texts, labels)):
    print(f"===== fold {fold+1} =====")
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(classes)).to(device)
    cw = torch.tensor([len(labels) / (len(classes) * labels.count(c)) for c in range(len(classes))], dtype=torch.float32).to(device)
    tr_ds = TextDataset([texts[i] for i in tr], [labels[i] for i in tr], tokenizer)
    va_ds = TextDataset([texts[i] for i in va], [labels[i] for i in va], tokenizer)
    best = train_fold(model, tokenizer, tr_ds, va_ds, device, cw)
    acc, f1 = evaluate(model, va_ds, device)
    results.append({"fold": fold + 1, "acc": acc, "macro_f1": f1})
    print(f"  fold{fold+1} final acc={acc:.4f} macroF1={f1:.4f} (best={best:.4f})")

print("\n===== 汇总 =====")
for r in results:
    print(r)
print(f"mean acc={sum(r['acc'] for r in results)/len(results):.4f}  mean macroF1={sum(r['macro_f1'] for r in results)/len(results):.4f}")
